# Notebook 02 — Data Preprocessing

Applies the full preprocessing pipeline to the datasets.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [3]:

import pandas as pd
import matplotlib.pyplot as plt
import re
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


In [4]:
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

True

In [5]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()


**Load raw dataset**

In [6]:

df = pd.read_csv("../data/df_isot.csv")
print('Raw sample:')
print(df.head())

Raw sample:
                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                                text subject  \
0  Donald Trump just couldn t wish all Americans ...    News   
1  House Intelligence Committee Chairman Devin Nu...    News   
2  On Friday, it was revealed that former Milwauk...    News   
3  On Christmas day, Donald Trump announced that ...    News   
4  Pope Francis used his annual Christmas Day mes...    News   

                date  label  
0  December 31, 2017      0  
1  December 31, 2017      0  
2  December 30, 2017      0  
3  December 29, 2017      0  
4  December 25, 2017      0  


**The Preprocessing Function**

In [7]:
stop_words = set(stopwords.words('english')) - {'not', 'never'}

def preprocess_text(text):
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = text.lower()                          # lowercase
    tokens = word_tokenize(text)                 # tokenize
    tokens = [w for w in tokens if w.isalpha()]  # remove punctuation
    tokens = [w for w in tokens if w not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(w, pos='v')  for w in tokens]   # lemmatize
    return ' '.join(tokens)

**Preprocessing the 'Text' Column of the Dataset**

In [8]:

df['clean_text'] = df['text'].apply(preprocess_text)
print(f'\nDataset after preprocessing: {len(df)} articles')


Dataset after preprocessing: 44898 articles


**Preprocess the 'Title' column**

In [9]:

df['clean_title'] = df['title'].apply(preprocess_text)

**Combine the 'Text' and 'Title' column into a Single Column**

In [10]:

df['full_text'] = df['clean_title'] + " " + df['clean_text']
print(df[['title', 'clean_title', 'text', 'clean_text', 'full_text']].head())

                                               title  \
0   Donald Trump Sends Out Embarrassing New Year’...   
1   Drunk Bragging Trump Staffer Started Russian ...   
2   Sheriff David Clarke Becomes An Internet Joke...   
3   Trump Is So Obsessed He Even Has Obama’s Name...   
4   Pope Francis Just Called Out Donald Trump Dur...   

                                         clean_title  \
0  donald trump send embarrass new year eve messa...   
1  drink brag trump staffer start russian collusi...   
2  sheriff david clarke become internet joke thre...   
3     trump obsess even obama name cod website image   
4    pope francis call donald trump christmas speech   

                                                text  \
0  Donald Trump just couldn t wish all Americans ...   
1  House Intelligence Committee Chairman Devin Nu...   
2  On Friday, it was revealed that former Milwauk...   
3  On Christmas day, Donald Trump announced that ...   
4  Pope Francis used his annual Christmas Day 

In [11]:
df['full_text'].value_counts()
df['label'].value_counts()

label
0    23481
1    21417
Name: count, dtype: int64

**Remove Duplicate from text**

In [12]:
df = df.drop_duplicates(subset='full_text', keep='first').reset_index(drop=True)


In [13]:
df['full_text'].value_counts()
df['label'].value_counts()

label
1    21041
0    17901
Name: count, dtype: int64

**Save Preprocessed Dataset for use in Later Notebooks**

In [14]:
df.to_csv('../data/preprocessed.csv', index=False)
print('Saved to ../data/processed/isot_preprocessed.csv')

Saved to ../data/processed/isot_preprocessed.csv


**Load LIAR test data and create binary labels**

In [15]:

liar_cols = ['id', 'label', 'statement', 'subject', 'speaker',
             'speaker_job', 'state', 'party', 'barely_true',
             'false', 'half_true', 'mostly_true', 'pants_fire', 'context']

liar_test = pd.read_csv('../data/test.tsv', sep='\t', header=None, names=liar_cols)

label_map = {
    'pants-fire'  : 0,
    'false'       : 0,
    'barely-true' : 0,
    'half-true'   : None,
    'mostly-true' : 1,
    'true'        : 1
}

liar_test = liar_test[liar_test['label'].isin(label_map)].copy()
liar_test['binary_label'] = liar_test['label'].map(label_map)
liar_test = liar_test[liar_test['binary_label'].notna()]

X_liar = liar_test['statement']
y_liar = liar_test['binary_label'].astype(int)

print(f"LIAR test samples : {len(liar_test)}")
print(y_liar.value_counts().rename({0:'fake', 1:'real'}))

LIAR test samples : 1002
binary_label
fake    553
real    449
Name: count, dtype: int64


**Preprocess Dataset and Drop empty rows after preprocessing**

In [16]:

X_liar = liar_test['statement'].apply(preprocess_text)

# Drop empty rows after preprocessing
mask   = X_liar.str.strip() != ''
X_liar = X_liar[mask]
y_liar = y_liar[mask]

print(f"After preprocessing: {len(X_liar)} samples")
print(X_liar.iloc[0])

After preprocessing: 1002 samples
build wall border take literally years


**Save preprocessed LIAR test set**

In [17]:

liar_test['clean_statement'] = X_liar
liar_test['binary_label']    = y_liar

liar_test[['clean_statement', 'binary_label']].to_csv('../data/liar_test_preprocessed.csv', index=False)
print('Saved to ../data/liar_test_preprocessed.csv')

Saved to ../data/liar_test_preprocessed.csv
